In [ ]:
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

from input import input
import config
from model import generics, single_ml_model_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

In [ ]:
# === CELULA DE CONFIGURACAO -- Tarefa 5 (fluxo notebook-only, mesmo padrao
# ja usado pelos notebooks de FS de ARIMA-MLP desde a Tarefa 3.2) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 5 do PLANO_ARQUITETURA.md: generalizacao do arsenal de Feature
# Selection (ja validado em ARIMA-MLP via Additive) para MLP single, via
# SKlearnModel -- confirmado por pre-check (Tarefa 5) que SKlearnModel e
# agnostico a identidade de `model` na mesma forma que Additive ja era
# (fit_predict_ml_schemma so chama model.fit/model.predict), entao o mesmo
# Pipeline([selector, estimator]) funciona sem NENHUMA mudanca em
# single_ml_model_exp.py/grid_search_exp.py. Diferente do hibrido, nao ha
# residuo do ARIMA nem copia de modelo linear pre-treinado -- SKlearnModel
# opera direto sobre a serie bruta (uma celula a menos que os notebooks de
# ARIMA-MLP: sem a celula de "copia do ARIMA").
#
# strategy fixa nesta execucao ('rf_embedded') -- numero de features decidido por SelectFromModel(threshold=None), SEM selector__k (Tarefa 3.1/3.9).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='rf_embedded')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# lag_size='auto' medido na Tarefa 5 (pre-check): resolve para os MESMOS
# valores que ja foram medidos para o hibrido ARIMA-MLP (airlines=20,
# austres=1, coloradoRiver=16, sunspot=9) -- confirmado com dado real, nao
# assumido (ver tests/model/test_single_ml_model_exp.py). df_train tem mais
# linhas que o hibrido (ex. airlines: 96 aqui vs. 80 no hibrido) porque
# SKlearnModel janela a serie bruta uma unica vez, sem o janelamento duplo
# implicito do residuo do ARIMA.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
# Convencao proposta na Tarefa 5 para distinguir a familia "MLP single" da
# familia "ARIMA-MLP hibrido" (que ja usa chamados_v4_fs_<metodo> sem "mlp"):
# chamados_v4_fs_mlp_<metodo>.
experiment_id = 'chamados_v4_fs_mlp_rfembedded'
# Caminhos derivados de experiment_id, computados uma vez aqui e reusados nas
# celulas 3/4/5 (mesmo padrao ja usado pelos notebooks de ARIMA-MLP).
experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py precisam disso (mesma regra ja usada por amv1<metodo> no
# hibrido). Baseline MLP single usa model_name='mlp' (sem prefixo de
# horizon, adicionado automaticamente por GridSearch/generics.format_names)
# -- aqui 'mlprfembedded' vira '1mlprfembedded' no nome do .pkl,
# mesma logica de 'amv1rfembedded' -> '1amv1rfembedded' no hibrido.
model_name = 'mlprfembedded'
normalize = True
force = False
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
model_parameters = {
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
   }

In [ ]:
# Sanity-check exigido pela Tarefa 2/5: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

# Checagem de identidade (mesmo padrao ja usado pelos notebooks de ARIMA-MLP,
# achado de code-review da Tarefa 3.2): trava que a 'strategy' do seletor
# (celula 1) e o experiment_id/model_name declarados na MESMA celula
# continuam consistentes entre si.
strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"experiment_id={experiment_id!r} -- confirme que voce nao mudou 'strategy' "
    "na celula de configuracao sem atualizar experiment_id/model_name."
)
assert strategy_slug in model_name, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"model_name={model_name!r} -- confirme consistencia."
)
print(f"OK -- strategy={model.named_steps['selector'].strategy!r} consistente com experiment_id={experiment_id!r} e model_name={model_name!r}")

In [ ]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST
# (mesmo motivo ja documentado nos notebooks de ARIMA-MLP). Diferente do
# hibrido, nao ha celula de "copia do ARIMA" -- SKlearnModel nao depende de
# nenhum modelo linear pre-treinado, opera direto sobre a serie bruta.
# use_val_slipt_for_prev=True e explicito porque o default de GridSearch
# (False) diverge do default do wrapper grid_seach_multiple_bases (True) --
# sem isso, o refit final perderia o val_size e os .pkl ficariam sem
# val_metrics, quebrando a paridade com o baseline original (mlp em
# chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

In [ ]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (mesmo padrao notebook-only ja usado pelo hibrido).
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = experiment_dir_results / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    experiment_dir,
    metrics_output,
    detail=True,
)
df_metrics

In [ ]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (mesmo padrao notebook-only ja usado pelo hibrido).
from utils.export_selected_features import run_export_selected_features

features_output = experiment_dir_results / 'selected_features.csv'
df_features = run_export_selected_features(
    experiment_dir,
    features_output,
    detail=True,
)
df_features